# 第 12 课：PyTorch CTCLoss——shape、length 与排错

目标：能够独立构造 `torch.nn.CTCLoss` 输入，理解 reduction、blank、无穷 loss 和梯度。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | CTC 核心 |
| 建议投入 | 4～6 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 11 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | CTCLoss shape、input/target lengths、zero_infinity |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：CTCLoss shape、input/target lengths、zero_infinity。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

import torch
import torch.nn.functional as F
torch.manual_seed(7)

项目根目录: G:\learn_asr


## 1. 四个输入及其形状

- `log_probs`: `[T, N, C]`
- `targets`: 拼接的一维标签，或 `[N,S]`
- `input_lengths`: 每个样本有效输出时间步
- `target_lengths`: 每个标签的有效长度

In [2]:
T,N,C=6,2,4
logits=torch.randn(T,N,C,requires_grad=True)
log_probs=logits.log_softmax(dim=-1)
targets=torch.tensor([1,2, 2,3,1],dtype=torch.long)
input_lengths=torch.tensor([6,5],dtype=torch.long)
target_lengths=torch.tensor([2,3],dtype=torch.long)
print("log_probs",log_probs.shape,"targets",targets.shape)
loss_fn=torch.nn.CTCLoss(blank=0,reduction="none",zero_infinity=False)
losses=loss_fn(log_probs,targets,input_lengths,target_lengths)
print("per-sample loss:",losses)

log_probs torch.Size([6, 2, 4]) targets torch.Size([5])
per-sample loss: tensor([6.2092, 3.7184], grad_fn=<CtcLossBackward0>)


## 2. Loss 就是负对数概率

`reduction="none"` 时，第一个样本的 `exp(-loss)` 就是目标文本所有合法路径的概率和。

In [3]:
print("P(target_0|X_0) =",torch.exp(-losses[0]).item())
loss=losses.mean(); loss.backward()
print("梯度 shape:",logits.grad.shape,"梯度是否有限:",torch.isfinite(logits.grad).all().item())

P(target_0|X_0) = 0.0020108476746827364
梯度 shape: torch.Size([6, 2, 4]) 梯度是否有限: True


## 3. 最常见的长度错误：下采样后忘记更新 input_lengths

In [4]:
def conv_len(L,k=3,s=2,p=1,d=1): return (L+2*p-d*(k-1)-1)//s+1
raw_lengths=torch.tensor([101,80,57])
after1=conv_len(raw_lengths); after2=conv_len(after1)
print("原帧数:",raw_lengths.tolist())
print("两层 stride=2 后:",after2.tolist())

原帧数: [101, 80, 57]
两层 stride=2 后: [26, 20, 15]


如果 logits 只有 26 步，却仍传入原始 101 步，CTCLoss 会报长度超过输入；如果时间轴被压得比标签合法最短路径还短，loss 会是 `inf`。

In [5]:
# 目标 [1,1] 至少需要 3 步；这里故意只给 2 步
bad_lp=torch.randn(2,1,3).log_softmax(-1)
bad_target=torch.tensor([1,1])
for zero in [False,True]:
    fn=torch.nn.CTCLoss(blank=0,reduction="none",zero_infinity=zero)
    print("zero_infinity=",zero,"loss=",fn(bad_lp,bad_target,torch.tensor([2]),torch.tensor([2])).item())

zero_infinity= False loss= inf
zero_infinity= True loss= 0.0


`zero_infinity=True` 能避免训练被 `inf` 污染，但它只是把不可能样本的 loss/gradient 置零；真正应该检查下采样比例、标签长度和数据。

## 4. 建立统一的 batch 检查器

In [6]:
def min_ctc_steps(seq):
    return len(seq)+sum(a==b for a,b in zip(seq,seq[1:]))

def audit_ctc_batch(input_lengths,target_list,num_classes,blank=0):
    problems=[]
    for i,(T,target) in enumerate(zip(input_lengths.tolist(),target_list)):
        if any(x==blank for x in target): problems.append((i,"target 中含 blank"))
        if any(x<0 or x>=num_classes for x in target): problems.append((i,"token id 越界"))
        need=min_ctc_steps(target)
        if T<need: problems.append((i,f"T={T} < 最少需要 {need}"))
    return problems

print(audit_ctc_batch(torch.tensor([4,2]),[[1,2],[1,1]],4))

[(1, 'T=2 < 最少需要 3')]


## 本课测试

1. 为什么 `log_probs` 要先做 `log_softmax`？
2. `[T,N,C]` 三个维度分别是什么？
3. blank 能否出现在训练 target 中？
4. `zero_infinity=True` 是否真正修复了数据？
5. 两层 stride=2 后，长度大约缩短多少倍？

<details><summary>展开参考答案</summary>

1. CTCLoss 接受对数概率。2. 时间、batch、类别。3. 不应出现。4. 没有，只是让不可能样本不破坏梯度。5. 约 4 倍，精确值应逐层套卷积长度公式。

</details>

## 下一课：Greedy 与 CTC Prefix Beam Search。

<!-- course-upgrade-v2 -->
## 强化练习：第 12 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `CTCLoss shape`、`input/target lengths`、`zero_infinity`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**下采样后 input_lengths 仍使用原长度**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**写 batch audit 检查重复 token 和合法最短 T**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**说明 loss 异常怎样定位到数据管线**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：CTCLoss shape、input/target lengths、zero_infinity。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 CTCLoss shape、input/target lengths、zero_infinity。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
